# Session 5 — Your turn: Finding structure without labels

No labels this time. You will choose the number of clusters from the data alone, and then see how many dimensions a set of images really needs.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np, pandas as pd
from sklearn.datasets import make_blobs, load_digits
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

X, hidden = make_blobs(n_samples=500, centers=5, cluster_std=[0.8, 1.2, 0.9, 1.0, 0.7], center_box=(-12, 12), random_state=11)
X_scaled = StandardScaler().fit_transform(X)
print(X.shape)
from aij_check import check, close, summary

(500, 2)


## Task 1: How many clusters?

Run k-means (`n_init=20`, `random_state=0`) for K from 2 to 8 on `X_scaled` and store each K's silhouette score in `scores` (a dictionary K -> score).
Set `best_k` to the K with the highest silhouette. Then compare it with the hidden truth: `agreement` is the adjusted Rand index between the k-means labels at `best_k` and `hidden`.

In [2]:
from sklearn.metrics import adjusted_rand_score
scores = {}
for k in range(2, 9):
    scores[k] = silhouette_score(X_scaled, KMeans(n_clusters=k, n_init=20, random_state=0).fit_predict(X_scaled))
best_k = max(scores, key=scores.get)
agreement = adjusted_rand_score(hidden, KMeans(n_clusters=best_k, n_init=20, random_state=0).fit_predict(X_scaled))
print(scores, best_k, agreement)

{2: 0.5082221923330497, 3: 0.6280960501803498, 4: 0.7604702586150102, 5: 0.7939854359261119, 6: 0.7192277529964124, 7: 0.626484702521542, 8: 0.5518202056566776} 5 1.0


In [3]:
check("scores for K = 2 to 8", lambda: sorted(scores) == list(range(2, 9)))
check("best_k has the highest silhouette", lambda: scores[best_k] == max(scores.values()))
check("the silhouette points at the five blobs", lambda: best_k == 5, "The data was generated around five centres.")
check("the clusters agree with the hidden truth", lambda: agreement > 0.9)
summary()

PASS  scores for K = 2 to 8
PASS  best_k has the highest silhouette
PASS  the silhouette points at the five blobs
PASS  the clusters agree with the hidden truth

4 of 4 checks passed.  Well done.


**Why it works.** The silhouette compares each point's distance to its own cluster with its distance to the nearest other one; it peaks when clusters are compact and separated. It is a clue, not a proof: always plot the clusters too.

## Task 2: How many dimensions do the digits need?

Standardise the 64 pixel features of `load_digits()` and fit a PCA with all components. Store the cumulative share of variance in `cumulative` (an array of length 64),
`n_for_90` for the smallest number of components that keep at least 90% of the variance, and `kept_by_two` for the share the first two components keep.

In [4]:
digits = load_digits()
X_digits = StandardScaler().fit_transform(digits.data)
pca = PCA().fit(X_digits)
cumulative = np.cumsum(pca.explained_variance_ratio_)
n_for_90 = int(np.searchsorted(cumulative, 0.90) + 1)
kept_by_two = float(cumulative[1])
print(n_for_90, kept_by_two)

31 0.21594970500832789


In [5]:
check("one cumulative value per component", lambda: len(cumulative) == 64 and abs(cumulative[-1] - 1) < 1e-9)
check("31 components keep 90% of the variance", lambda: n_for_90 == 31)
check("two components keep about 22%", lambda: close(kept_by_two, 0.216, 0.005))
summary()

PASS  one cumulative value per component
PASS  31 components keep 90% of the variance
PASS  two components keep about 22%

3 of 3 checks passed.  Well done.


**Why it works.** A 2D map of the digits keeps only about a fifth of the variance, which is why the clusters overlap there. It takes about half of the 64 components to keep 90%, so the digits have no simple low-dimensional shape.